In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_samples

df = pd.read_csv("../data/processed/customers_clean.csv")
pipeline = joblib.load("../models/customeriq_kmeans_pipeline.joblib")

FEATURES = ["Number_of_Orders", "Avg_Order_Value", "Days_Since_Last_Purchase",
            "Items_Per_Order", "Average_Rating", "Discount_Applied"]

X_scaled = pipeline.named_steps["preprocess"].transform(df[FEATURES])
base_labels = pipeline.predict(df[FEATURES])
print(X_scaled.shape)

(60000, 6)


In [2]:
#how well-separated are the clusters?
rng = np.random.default_rng(42)
idx = rng.choice(len(X_scaled), 10000, replace=False)
sil = silhouette_samples(X_scaled[idx], base_labels[idx])

print("Overall silhouette:", sil.mean().round(3))
print()
print(pd.Series(sil).groupby(base_labels[idx]).mean().round(3))
print()
print("Share of customers with a negative score:", (sil < 0).mean().round(3))

Overall silhouette: 0.25

0    0.207
1    0.248
2    0.319
3    0.236
dtype: float64

Share of customers with a negative score: 0.011


In [ ]:
results = []
for seed in range(1, 11):
    km = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=seed).fit(X_scaled)
    results.append({
        "seed": seed,
        "inertia": round(km.inertia_),
        "ARI_vs_final": round(adjusted_rand_score(base_labels, km.labels_), 3),
    })
pd.DataFrame(results)
#This retrains K-Means 10 times, each with a different random seed, and compares each result with our saved model.

,seed,inertia,ARI_vs_final
0,1,175664,0.999
1,2,175665,1.000
2,3,175664,0.998
3,4,175665,1.000
4,5,175665,1.000
5,6,175664,0.999
6,7,175665,0.999
7,8,175664,0.998
8,9,175665,1.000
9,10,175665,1.000


In [4]:
single_runs = []
for seed in range(1, 11):
    km = KMeans(n_clusters=4, init="random", n_init=1, random_state=seed).fit(X_scaled)
    single_runs.append({
        "seed": seed,
        "inertia": round(km.inertia_),
        "ARI_vs_final": round(adjusted_rand_score(base_labels, km.labels_), 3),
    })
pd.DataFrame(single_runs)

,seed,inertia,ARI_vs_final
0,1,175665,1.000
1,2,175667,0.992
2,3,175665,0.999
3,4,175665,1.000
4,5,175665,1.000
5,6,175665,1.000
6,7,175665,1.000
7,8,175665,1.000
8,9,175667,0.992
9,10,175665,0.999


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.metrics import silhouette_score

LOG_FEATURES = ["Number_of_Orders", "Avg_Order_Value", "Days_Since_Last_Purchase"]

def run_experiment(plain_features, k=4):
    pre = ColumnTransformer([
        ("log_features", Pipeline([("log", FunctionTransformer(np.log1p)),
                                   ("scale", StandardScaler())]), LOG_FEATURES),
        ("plain_features", Pipeline([("impute", SimpleImputer(strategy="median")),
                                     ("scale", StandardScaler())]), plain_features),
    ])
    X = pre.fit_transform(df[LOG_FEATURES + plain_features])
    labels = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42).fit_predict(X)
    sil = silhouette_score(X, labels, sample_size=10000, random_state=42)
    return labels, round(sil, 3)

In [6]:
base_plain = ["Items_Per_Order", "Average_Rating", "Discount_Applied"]

labels_base, sil_base = run_experiment(base_plain)
labels_no_disc, sil_no_disc = run_experiment(["Items_Per_Order", "Average_Rating"])
labels_age, sil_age = run_experiment(base_plain + ["Age"])

print("Baseline (6 features):", sil_base)
print("Without Discount:     ", sil_no_disc,
      "| agreement with baseline (ARI):", round(adjusted_rand_score(labels_base, labels_no_disc), 3))
print("With Age added:       ", sil_age,
      "| agreement with baseline (ARI):", round(adjusted_rand_score(labels_base, labels_age), 3))

print()
print("Without Discount, cluster sizes:", pd.Series(labels_no_disc).value_counts(normalize=True).round(2).sort_index().tolist())
print(pd.crosstab(labels_base, labels_no_disc))

Baseline (6 features): 0.247
Without Discount:      0.256 | agreement with baseline (ARI): 0.463
With Age added:        0.227 | agreement with baseline (ARI): 0.63

Without Discount, cluster sizes: [0.23, 0.17, 0.26, 0.35]
col_0     0     1      2      3
row_0                          
0        71     0  13507    148
1      6578   180   1160  12993
2         0  9851      0   1983
3      6961     8    973   5587


## Evaluation Summary (K = 4)

**There is no accuracy for clustering.** There are no true labels, so I judged the model on separation, stability, size balance and interpretability.

| Check | Result |
|---|---|
| Overall silhouette (10,000-customer sample) | 0.25: groups overlap, with soft edges |
| Silhouette by cluster | Champions 0.319 (tightest), Discount-Driven Regulars 0.248, At-Risk Full-Price Buyers 0.236, Occasional Low-Spenders 0.207 (fuzziest) |
| Customers closer to another cluster than their own | 1.1% |
| Stability, 10 random seeds with k-means++ and n_init=10 | ARI 0.998–1.000, inertia 175,664–175,665 |
| Worst case: random init, single run, 10 seeds | ARI 0.992–1.000 |
| Cluster sizes | 19.7%–34.9%: no tiny or dominant cluster |

**Sensitivity tests**
- Removing `Discount_Applied` gives ARI 0.463 against the baseline. Champions and Occasional Low-Spenders mostly persist, but Discount-Driven Regulars and At-Risk Full-Price Buyers are separated largely by discount usage.
- Adding `Age` gives ARI 0.63 and a lower silhouette (0.227 vs 0.247), so Age was kept out of the model and used only for describing clusters.

**Limitations**
- Clusters overlap (silhouette about 0.25), so customers near a boundary could reasonably belong to either group.
- Stable results do not mean well-separated: the algorithm finds the same groups each time, but the groups touch.
- The segmentation depends on feature choices, especially the yes/no discount flag.
- The dataset is synthetic, so the findings demonstrate the method and say nothing about real customers.